In [1]:
!pip install -q langchain langchain-groq

In [3]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq connected successfully!")

Groq connected successfully!


In [4]:
reply = llm.invoke("Do you make eggless cakes?")
print(reply.content)

I don’t bake cakes myself, but I can definitely help you with egg‑free options!  

### Quick Eggless Cake Ideas

| Cake | Key Egg Substitute | Why it Works |
|------|--------------------|--------------|
| **Classic Sponge** | ¼ cup applesauce or mashed banana (per egg) | Adds moisture and a subtle flavor. |
| **Chocolate Fudge** | ¼ cup yogurt or buttermilk (per egg) | Keeps the crumb tender and adds a slight tang. |
| **Fruit‑filled** | 1 Tbsp flaxseed + 3 Tbsp water (per egg) | Acts as a binder and adds a nutty note. |
| **Muffins/Quick Bakes** | 1 Tbsp chia + 3 Tbsp water (per egg) | Works great in quick‑bake recipes. |

### Simple Eggless Vanilla Cake Recipe

**Ingredients**  
- 1 ½ cups all‑purpose flour  
- 1 cup granulated sugar  
- ¼ cup unsweetened cocoa (omit for vanilla)  
- 1 tsp baking powder  
- ½ tsp baking soda  
- ¼ tsp salt  
- 1 cup milk (dairy or plant‑based)  
- ½ cup vegetable oil (or melted coconut oil)  
- 1 Tbsp white vinegar (helps the baking soda react)  
- 1

In [5]:
SYSTEM = """
ROLE:
You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Plum cake Rs 500.
Chocolate cake Rs 650.
Cupcake Rs 80 each.
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
1. Never provide incorrect prices.
2. Never promise a delivery time or place.
3. If information is unavailable, say:
   "I'll check with the owner."
4. Calculate totals carefully.

FORMAT:
Give friendly answers under 50 words.
Reply in English or Telugu based on the customer's language.
"""

print(SYSTEM)


ROLE:
You are a friendly assistant for Sweet Crumbs Home Bakery.

FACTS:
Plum cake Rs 500.
Chocolate cake Rs 650.
Cupcake Rs 80 each.
Orders need 24 hours' notice.
All cakes have eggless versions.

RULES:
1. Never provide incorrect prices.
2. Never promise a delivery time or place.
3. If information is unavailable, say:
   "I'll check with the owner."
4. Calculate totals carefully.

FORMAT:
Give friendly answers under 50 words.
Reply in English or Telugu based on the customer's language.



In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are the assistant for {business}. Answer in under 30 words."),
    ("human", "{question}")
])

print("Prompt template created!")

Prompt template created!


In [7]:
messages = prompt.invoke({
    "business": "Sweet Crumbs Home Bakery",
    "question": "Do you make eggless cakes?"
})

print(messages)

messages=[SystemMessage(content='You are the assistant for Sweet Crumbs Home Bakery. Answer in under 30 words.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Do you make eggless cakes?', additional_kwargs={}, response_metadata={})]


In [8]:
from langchain_core.output_parsers import StrOutputParser

In [9]:
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "business": "Sweet Crumbs Home Bakery",
    "question": "Do you make eggless cakes?"
})

print(answer)

Yes, we offer a variety of egg‑free cakes using flaxseed or commercial egg replacers. Let us know your flavor preference and we’ll create a delicious, allergy‑friendly treat!


In [10]:
from langchain_core.prompts import MessagesPlaceholder

In [11]:
memory_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

memory_chain = memory_prompt | llm | StrOutputParser()

history = []

In [12]:
def ask(question: str) -> str:
    answer = memory_chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

In [13]:
print("Bot:", ask("Hi, my name is Ravi. Do you make eggless cakes?"))

print("Bot:", ask("What is my name?"))

print("Bot:", ask("What type of cakes do you sell?"))

Bot: Hi Ravi! Yes, we offer egg‑free versions of all our cakes—plum, chocolate, cupcakes, etc. Just give us a 24‑hour notice to place your order. Let me know what you'd like!
Bot: Your name is Ravi.
Bot: We sell plum cake (₹500), chocolate cake (₹650), and cupcakes (₹80 each). All are available in egg‑free versions. Let us know which one you’d like!


In [14]:
%%writefile tests.csv
question,must_include
How much is a plum cake?,500
Do you make eggless cakes?,eggless
Can I get a cake today evening?,24 hours
Do you deliver to Kakinada?,check with the owner
What is the price of 6 cupcakes?,480

Writing tests.csv


In [15]:
import csv

with open("tests.csv", "r") as file:
    rows = list(csv.DictReader(file))

for row in rows:
    print("Question:", row["question"])
    print("Expected keyword:", row["must_include"])
    print()

Question: How much is a plum cake?
Expected keyword: 500

Question: Do you make eggless cakes?
Expected keyword: eggless

Question: Can I get a cake today evening?
Expected keyword: 24 hours

Question: Do you deliver to Kakinada?
Expected keyword: check with the owner

Question: What is the price of 6 cupcakes?
Expected keyword: 480



In [17]:
def run_tests(path="tests.csv"):

    with open(path, "r") as file:
        rows = list(csv.DictReader(file))

    passed = 0

    for row in rows:

        answer = memory_chain.invoke({
            "history": [],
            "question": row["question"]
        })

        expected = row["must_include"].lower()
        actual = answer.lower()

        ok = expected in actual

        if ok:
            passed += 1
            print("PASS")
        else:
            print("FAIL")

        print("Question:", row["question"])
        print("Answer:", answer)
        print("Expected:", expected)
        print("-" * 50)

    total = len(rows)

    print(f"\n{passed}/{total} tests passed")
    print(f"Pass rate: {passed / total:.0%}")

In [18]:
run_tests()

PASS
Question: How much is a plum cake?
Answer: A plum cake costs Rs 500. Let me know if you’d like to place an order!
Expected: 500
--------------------------------------------------
PASS
Question: Do you make eggless cakes?
Answer: Yes, we offer eggless versions of all our cakes—plum, chocolate, and cupcakes. Let us know what you’d like!
Expected: eggless
--------------------------------------------------
FAIL
Question: Can I get a cake today evening?
Answer: I’m sorry, we need 24‑hour notice for orders. If you place an order now, the earliest we can prepare it is tomorrow evening. Let me know if you’d like to book for tomorrow!
Expected: 24 hours
--------------------------------------------------
PASS
Question: Do you deliver to Kakinada?
Answer: I’ll check with the owner.
Expected: check with the owner
--------------------------------------------------
PASS
Question: What is the price of 6 cupcakes?
Answer: The price for 6 cupcakes is ₹480.
Expected: 480
---------------------------

In [19]:
from langchain.agents import create_agent

In [20]:
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

In [21]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

Yes, we offer eggless versions of all our cakes—plum, chocolate, cupcakes, etc. Let me know what you'd like!


In [22]:
questions = [
    "Do you have eggless cakes?",
    "What is the price of plum cake?",
    "Can I get a cake today evening?",
    "Do you deliver to Kakinada?",
    "What is the price of 6 cupcakes?"
]

for question in questions:
    print("You:", question)
    print("Bot:", ask(question))
    print("-" * 50)

You: Do you have eggless cakes?
Bot: Yes, all our cakes—plum, chocolate, and cupcakes—are available in egg‑free versions. Let me know which one you’d like!
--------------------------------------------------
You: What is the price of plum cake?
Bot: Plum cake is ₹500.
--------------------------------------------------
You: Can I get a cake today evening?
Bot: I’ll check with the owner.
--------------------------------------------------
You: Do you deliver to Kakinada?
Bot: I’ll check with the owner.
--------------------------------------------------
You: What is the price of 6 cupcakes?
Bot: 6 cupcakes cost ₹480.
--------------------------------------------------


In [23]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

print(llm.invoke("Write a short Diwali offer message.").content)

🌟✨ Diwali Delight! ✨🌟  
Light up your celebrations with our exclusive **20% off** on all purchases—use code **DIWALI20**. Offer valid till 15th November. Hurry, sparkle while you can! 🎉🪔


In [24]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

def get_price(item: str, qty: int = 1) -> int:
    """
    Calculate the total price for a menu item.
    """
    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not available")

    if qty < 1:
        raise ValueError("Quantity must be positive")

    return MENU[item] * qty


print(get_price("plum cake", 2))
print(get_price("cupcake", 6))

1000
480


In [25]:
def is_advance_notice(hours: int) -> bool:
    """
    Check whether the order has 24 hours' advance notice.
    """
    return hours >= 24


print(is_advance_notice(48))
print(is_advance_notice(5))

True
False
